In [ ]:
# update of 'Empirical_MPRA_v_Malinois_Scatters_v4.ipynb' for revision
# includes indels now for the following updated figures
# Figure 1B (activity correlation)
# Figure 1C (emVar correlation)
# Figure 1E UKBB PRC (formerly in 'sklearn_precision_recall_no_act_sei_all_v4.py'
# Supplemental Figure 3 (indel alone correlations)
# Reviewer Response Figures for R3C2
# Supplemental Figures 11 + 12 (PRCs)

In [1]:
# import packages
import pandas as pd
import os
from tqdm import tqdm
from scipy import stats
import torch
import matplotlib.pyplot as mpl
import seaborn as sns
import numpy as np
import matplotlib
from collections import Counter
import matplotlib.patches as mpatches

In [2]:
# functions needed to convert raw prediction files to DF for plotting
# only taking the forward strand prediction for comparison to MPRA 
# similarly, only using the variant at position 100 to match MPRA
def splitter (string):
    split = string[1:-1].split(',')
    float_split = [float(i) for i in split]
    return float_split

In [3]:
# define function to convert pt to DF
def pt2df (path2pt, n_preds):
    # Open predictions
    pt = torch.load(path2pt)
    # Open VCF used for generating predictions
    vcf = pt['vcf']
    # Use VCF to generate collapsed IDs chr:pos:ref:alt:INFO
    vcf_ids = [(':').join([i,str(j),k,l]) for i,j,k,l in zip(vcf['chrom'],
                                                             vcf['pos'],
                                                             vcf['ref'],
                                                             vcf['alt'])]
    # Make Dictionary of ID:number (basically index) pairs
    id_dict = dict(zip(vcf_ids, [i for i in range(0,len(vcf_ids))]))
    # Iterate over Dictionary to combine ID from VCF with predictions from Tensor
    df_rows = []
    for i in tqdm(id_dict):
        tmp = []
        # Split ID back into Chrom, Pos, Ref, Alt
        splitted = i.split(':')
        chrom = splitted[0]
        pos = splitted[1]
        ref = splitted[2]
        alt = splitted[3]
        # Make a tmp list for ref/alt each cell type
        # K562
        k562_ref_list = []
        k562_alt_list = []
        # HepG2
        hepg2_ref_list = []
        hepg2_alt_list = []
        # SK-N-SH
        sknsh_ref_list = []
        sknsh_alt_list = []
        # Iterate of each prediction
        for x in range(0, n_preds):
            # Get reference for predictions for fwd and rev strands
            ref_pred = pt['ref'][id_dict.get(i), 0, x]
            #ref_rev_comp = pt['ref'][id_dict.get(i), 1, x]
            # Take average of fwd/rev_comp predictions and add to cell type list
            k562_ref_list.append(ref_pred[0].item())
            hepg2_ref_list.append(ref_pred[1].item())
            sknsh_ref_list.append(ref_pred[2].item())
            # Get alternate predictions for fwd and rev strands
            alt_pred = pt['alt'][id_dict.get(i), 0, x]
            #alt_rev_comp = pt['alt'][id_dict.get(i), 1, x]
            # Take average of fwd/rev_comp predictions and add to cell type list
            k562_alt_list.append(alt_pred[0].item())
            hepg2_alt_list.append(alt_pred[1].item())
            sknsh_alt_list.append(alt_pred[2].item())
        # Subtract alt-ref to get skew
        k562_skew = [j-i for i,j in zip(k562_ref_list, k562_alt_list)]
        hepg2_skew = [j-i for i,j in zip(hepg2_ref_list, hepg2_alt_list)]
        sknsh_skew = [j-i for i,j in zip(sknsh_ref_list, sknsh_alt_list)]
        # Build row as tmp list
        tmp.append(chrom)
        tmp.append(pos)
        tmp.append(ref)
        tmp.append(alt)
        tmp.append(k562_ref_list)
        tmp.append(k562_alt_list)
        tmp.append(hepg2_ref_list)
        tmp.append(hepg2_alt_list)
        tmp.append(sknsh_ref_list)
        tmp.append(sknsh_alt_list)
        tmp.append(k562_skew)
        tmp.append(hepg2_skew)
        tmp.append(sknsh_skew)
        # Append tmp list to "rows" list for formatting to DF
        df_rows.append(tmp)
    col_names = ['chrom',
                 'pos',
                 'ref',
                 'alt',
                 'k562_ref_pred',
                 'k562_alt_pred',
                 'hepg2_ref_pred',
                 'hepg2_alt_pred',
                 'sknsh_ref_pred',
                 'sknsh_alt_pred',
                 'k562_skew_pred',
                 'hepg2_skew_pred',
                 'sknsh_skew_pred']
    df = pd.DataFrame(df_rows, columns=col_names)
    return df

In [4]:
def pt2df_dict (path2pts, n_preds):
    df_dict = {}
    pt_list = [i for i in os.listdir(path2pts) if i.endswith('.pt')]
    for i in pt_list:
        print(i)
        chr_key = i.split('_')[0]
        df_dict.update({chr_key : pt2df(f'{path2pts}/{i}', n_preds)})
    return df_dict

In [5]:
# open gtex 017 preds
gtex_017_dict = pt2df_dict('../raw_data/gtex_snp_raw_preds/',
                           18)
# open traits 017 preds
traits_017_dict = pt2df_dict('../raw_data/ukbb_snp_raw_preds/',
                             18)

chr10_all_gtex_017.pt


100%|██████████| 9783/9783 [00:05<00:00, 1887.34it/s]


chr8_all_gtex_017.pt


100%|██████████| 8046/8046 [00:04<00:00, 1871.41it/s]


chr17_all_gtex_017.pt


...

100%|██████████| 8630/8630 [00:04<00:00, 2025.22it/s]


chr21_traits_hg38_017.pt


100%|██████████| 1518/1518 [00:00<00:00, 2055.14it/s]


chr22_traits_hg38_017.pt


100%|██████████| 2680/2680 [00:01<00:00, 2067.34it/s]


In [6]:
# convert to DF with column of pos 100 score ref/alt, avg all 18 ref/alt, and max ref/alt
# concatenate gtex into single DF
gtex_017_preds_df = pd.concat(gtex_017_dict.values())
# concatenate traits into single DF
traits_017_preds_df = pd.concat(traits_017_dict.values())

In [7]:
# define function to add columns listed in line 119
def add_100_avg(pt_df, one_hundred_index):
    # list of column names
    col_names = ['k562_ref_pred', 'k562_alt_pred',
                 'hepg2_ref_pred', 'hepg2_alt_pred',
                 'sknsh_ref_pred', 'sknsh_alt_pred',
                 'k562_skew_pred', 'hepg2_skew_pred', 'sknsh_skew_pred']
    # add position 100
    for i in tqdm(col_names):
        pt_df.loc[:,][f'{i}_pos_100'] = [
            float(j[one_hundred_index]) for j in pt_df[i]]
    # add avg all predictions
    for i in tqdm(col_names):
        pt_df.loc[:,][f'{i}_avg'] = [np.mean(j) for j in pt_df[i]]
    return pt_df

In [8]:
# add avg and center scores to gtex and traits concatenated DFs
# gtex
gtex_017_preds_df = add_100_avg(gtex_017_preds_df,
                                8)
# add ID to above DF
gtex_017_preds_df.loc[:,]['id'] = [(':').join([i, str(j), k, l]) for i, j, k, l in zip(gtex_017_preds_df['chrom'],
                                                                                       gtex_017_preds_df['pos'],
                                                                                       gtex_017_preds_df['ref'],
                                                                                       gtex_017_preds_df['alt'])]
# traits
traits_017_preds_df = add_100_avg(traits_017_preds_df,
                                  8)
# add ID to above DF
traits_017_preds_df.loc[:,]['id'] = [(':').join([i, str(j), k, l]) for i, j, k, l in zip(traits_017_preds_df['chrom'],
                                                                                         traits_017_preds_df['pos'],
                                                                                         traits_017_preds_df['ref'],
                                                                                         traits_017_preds_df['alt'])]
# combine gtex and traits into single df
all_gtex_traits_preds = pd.concat([gtex_017_preds_df, traits_017_preds_df])

100%|██████████| 9/9 [00:09<00:00,  1.09s/it]


In [9]:
# Open blacklist filtered MPRA data
all_mpra_filtered = pd.read_csv('../processed_data/all.gtex.traits.mpra.blacklist.filtered.v2.txt',
                                sep = '\t',
                                low_memory=False).drop_duplicates(subset='A_log2FC')
# merge and mpra data for snps
all_merge = all_mpra_filtered.merge(all_gtex_traits_preds, 
                                      how = 'inner', 
                                      left_on='hg38_id',
                                      right_on='id').drop_duplicates(subset='A_log2FC')
# filter for only snps
snps_mask = (all_merge['allele1'].str.len() == 1) & (all_merge['allele2'].str.len() == 1)
all_merge_snps = all_merge.loc[snps_mask].copy()
# add controls back in
controls = all_merge[all_merge['cohort'] == 'control'].copy()
controls.loc[:,'ref'] = [i.split(':')[-2] for i in controls['variant']]
controls.loc[:,'alt'] = [i.split(':')[-1] for i in controls['variant']]
controls.loc[:,'chromosome'] = [i.split(':')[0] for i in controls['variant']]
# drop non-snp controls
snp_control_mask = (controls['ref'].str.len() == 1) & (controls['alt'].str.len() == 1)
snp_controls = controls.loc[snp_control_mask]
# concatenate controls and snps
all_snps_with_controls = pd.concat([all_merge_snps, snp_controls]).reset_index()

In [10]:
def pt2df_dict_indel (path2pts, n_preds):
    df_dict = {}
    pt_list = [i for i in os.listdir(path2pts) if i.endswith('.pt')]
    for i in pt_list:
        print(i)
        chr_key = i.split('_')[3]
        df_dict.update({chr_key : pt2df(f'{path2pts}/{i}', n_preds)})
    return df_dict

In [11]:
# open the indel predictions
# open raw predictions
raw_indels_dict = pt2df_dict_indel('../raw_data/updated_indel_preds/',
                                   18)
raw_indels_df = pd.concat(raw_indels_dict.values())

updated_ukbb_gtex_chr6_raw_indel_preds.pt


100%|██████████| 925/925 [00:00<00:00, 2058.62it/s]


updated_ukbb_gtex_chr5_raw_indel_preds.pt


100%|██████████| 960/960 [00:00<00:00, 2072.70it/s]


updated_ukbb_gtex_chr9_raw_indel_preds.pt


...

100%|██████████| 1007/1007 [00:00<00:00, 2079.93it/s]


updated_ukbb_gtex_chr3_raw_indel_preds.pt


100%|██████████| 1072/1072 [00:00<00:00, 2085.31it/s]


updated_ukbb_gtex_chr4_raw_indel_preds.pt


100%|██████████| 851/851 [00:00<00:00, 2022.89it/s]


In [12]:
# open blacklist filtered mpra data
mpra_data = pd.read_csv('../processed_data/all.gtex.traits.mpra.blacklist.filtered.v2.txt', sep = '\t', low_memory=False)
# add avg and center scores to concatenated DF
raw_indels_df = add_100_avg(raw_indels_df,
                           8) # index of window with position variant centered at position 100
# add id
raw_indels_df.loc[:,'id'] = [(':').join([i, str(j), k, l]) for i, j, k, l in zip(raw_indels_df['chrom'],
                                                                                 raw_indels_df['pos'],
                                                                                 raw_indels_df['ref'],
                                                                                 raw_indels_df['alt'])]
# filter empirical data for merging predictions
mpra_indels = mpra_data[mpra_data['hg38_id'].isin(raw_indels_df['id'].tolist())]
# merge with experimental data
raw_merge = mpra_indels.merge(raw_indels_df,
                              how = 'inner',
                              left_on='hg38_id',
                              right_on='id').drop_duplicates(subset=['A_log2FC'])
# filter for only insertions/deletions <= 10bp
indel_len_bool = [True if len(ref) <= 10 and len(alt) <= 10 else False for ref, alt in zip(raw_merge['ref'],
                                                                                         raw_merge['alt'])]
raw_less_ten = raw_merge.loc[indel_len_bool]

100%|██████████| 9/9 [00:01<00:00,  7.13it/s]


In [13]:
# concatenate the dfs for plotting figure 1B and 1C
snps_and_indels_merge = pd.concat([all_snps_with_controls, raw_less_ten])
# save this combined df (we have matched the correct prediction pipeline, ie vcf_predict_indel with updated handling)
snps_and_indels_merge.to_csv('../processed_data/all.gtex.traits.mpra.blacklist.filtered.v2.with.updated.indel.preds.tsv',
                             sep = '\t',
                             index = False)

In [14]:
snps_and_indels_merge.keys()

Index(['index', 'hg38_id', 'variant', 'allele1', 'allele2', 'cohort',
       'cell_type', 'A_log2FC', 'B_log2FC', 'Log2Skew', 'emVar', 'chromosome',
       'chrom', 'pos', 'ref', 'alt', 'k562_ref_pred', 'k562_alt_pred',
       'hepg2_ref_pred', 'hepg2_alt_pred', 'sknsh_ref_pred', 'sknsh_alt_pred',
       'k562_skew_pred', 'hepg2_skew_pred', 'sknsh_skew_pred',
       'k562_ref_pred_pos_100', 'k562_alt_pred_pos_100',
       'hepg2_ref_pred_pos_100', 'hepg2_alt_pred_pos_100',
       'sknsh_ref_pred_pos_100', 'sknsh_alt_pred_pos_100',
       'k562_skew_pred_pos_100', 'hepg2_skew_pred_pos_100',
       'sknsh_skew_pred_pos_100', 'k562_ref_pred_avg', 'k562_alt_pred_avg',
       'hepg2_ref_pred_avg', 'hepg2_alt_pred_avg', 'sknsh_ref_pred_avg',
       'sknsh_alt_pred_avg', 'k562_skew_pred_avg', 'hepg2_skew_pred_avg',
       'sknsh_skew_pred_avg', 'id'],
      dtype='object')

In [ ]:
len(snps_and_indels_merge['id'].unique())

In [ ]:
# define a function to refactor data (split ref/alt into separate rows) for plotting activities more easily in other script
def refactor_activities (cell_type,
                         merged_preds,
                         score_strat):
    # filter for only cell type
    cell_df = merged_preds[merged_preds['cell_type'] == cell_type].drop_duplicates(subset='variant')
    # split into ref only
    cell_ref = cell_df.filter(['variant', 'hg38_id', 'A_log2FC', f'{cell_type.lower()}_ref_pred_{score_strat}'])
    # rename empirical MPRA column
    cell_ref['empirical_l2fc'] = cell_ref['A_log2FC']
    # rename predicted MPRA column
    cell_ref['predicted_l2fc'] = cell_ref[f'{cell_type.lower()}_ref_pred_{score_strat}']
    # filter for merging
    ref_reformat = cell_ref.filter(['variant', 'hg38_id', 'empirical_l2fc', 'predicted_l2fc'])
    # split into alt only
    cell_alt = cell_df.filter(['variant', 'hg38_id', 'B_log2FC', f'{cell_type.lower()}_alt_pred_{score_strat}'])
    # rename empirical MPRA column
    cell_alt['empirical_l2fc'] = cell_alt['B_log2FC']
    # rename predicted MPRA column
    cell_alt['predicted_l2fc'] = cell_alt[f'{cell_type.lower()}_alt_pred_{score_strat}']
    # filter for merginng
    alt_reformat = cell_alt.filter(['variant', 'hg38_id', 'empirical_l2fc', 'predicted_l2fc'])
    # combine 
    con = pd.concat([ref_reformat, alt_reformat])
    print(len(con))
    print(stats.pearsonr(x=con['empirical_l2fc'], y=con['predicted_l2fc']))
    print(stats.spearmanr(a=con['empirical_l2fc'], b=con['predicted_l2fc']))
    return con

In [ ]:
# get the correlation of activity predictions for everything together
# k562
k_act_refactor_all = refactor_activities('K562', snps_and_indels_merge, 'pos_100')
# hepg2
h_act_refactor_all = refactor_activities('HEPG2', snps_and_indels_merge, 'pos_100')
# sknsh
s_act_refactor_all = refactor_activities('SKNSH', snps_and_indels_merge, 'pos_100')

In [ ]:
# define function to plot skew correlations for emVars
def emvar_skew_scatter (merged_df, 
                        cell_type,
                        score_strat):
    # get cell type
    if cell_type == 'K562':
        cell_lower = 'k562'
    elif cell_type == 'HEPG2':
        cell_lower = 'hepg2'
    elif cell_type == 'SKNSH':
        cell_lower = 'sknsh'
    # filter for given cell t
    # filter for cell type
    cell_df = merged_df[(merged_df['cell_type'] == cell_type) &
                        (merged_df['emVar'] == True)]
    # get pearson r
    pearson, pval = stats.pearsonr(x=cell_df['Log2Skew'], 
                                   y=cell_df[f'{cell_lower}_skew_pred_{score_strat}'])
    print(pearson)
    print(pval)
    print(stats.spearmanr(a=cell_df['Log2Skew'], b=cell_df[f'{cell_lower}_skew_pred_{score_strat}']))
    print(len(cell_df))
    pearson = round(pearson, 2)
    return cell_df

In [ ]:
# filter full data for only emVars for plotting skew
# average of preds
k562_all_emvar_avg = emvar_skew_scatter(snps_and_indels_merge, 'K562', 'avg')
hepg2_all_emvar_avg = emvar_skew_scatter(snps_and_indels_merge, 'HEPG2', 'avg')
sknsh_all_emvar_avg = emvar_skew_scatter(snps_and_indels_merge, 'SKNSH', 'avg')

In [ ]:
### get correlations for annotating plots ###
### position 100 prediction activity ###
# k562
k_100_all_pearson = round(stats.pearsonr(x=k_act_refactor_all['empirical_l2fc'],
                                         y=k_act_refactor_all['predicted_l2fc'])[0], 2)
# hepg2
h_100_all_pearson = round(stats.pearsonr(x=h_act_refactor_all['empirical_l2fc'],
                                         y=h_act_refactor_all['predicted_l2fc'])[0] ,2)
# sknsh
s_100_all_pearson = round(stats.pearsonr(x=s_act_refactor_all['empirical_l2fc'],
                                         y=s_act_refactor_all['predicted_l2fc'])[0], 2)
### correlation emVar
# k562
k_all_emvar_pearson = round(stats.pearsonr(k562_all_emvar_avg['Log2Skew'],
                                           k562_all_emvar_avg['k562_skew_pred_avg'])[0], 2)
# hepg2
h_all_emvar_pearson = round(stats.pearsonr(hepg2_all_emvar_avg['Log2Skew'],
                                           hepg2_all_emvar_avg['hepg2_skew_pred_avg'])[0], 2)
# sknsh
s_all_emvar_pearson = round(stats.pearsonr(sknsh_all_emvar_avg['Log2Skew'],
                                           sknsh_all_emvar_avg['sknsh_skew_pred_avg'])[0], 2)

In [ ]:
f = mpl.figure(figsize=(6,8),dpi=300)
gs = f.add_gridspec(3,2)
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
### plot activities in column 1 ###
# sknsh activity #
with sns.axes_style('white'):
    ax_01 = f.add_subplot(gs[2,0])
    ax_01.set(adjustable='box', aspect='equal')
    mpl.plot([(.8 * min(s_act_refactor_all['empirical_l2fc'])), (.8 * max(s_act_refactor_all['empirical_l2fc']))],
             [(.8 * min(s_act_refactor_all['empirical_l2fc'])), (.8 * max(s_act_refactor_all['empirical_l2fc']))],
             linestyle='dashed',
             alpha=.5,
             color='k',
             linewidth=1)
    sns.scatterplot(data=s_act_refactor_all,
                    x='empirical_l2fc',
                    y='predicted_l2fc',
                    color = '#ED1C24',
                    s=1,
                    alpha=.1,
                    rasterized=True)
    sns.kdeplot(data=s_act_refactor_all, x='empirical_l2fc', y='predicted_l2fc',color='#ED1C24')
    mpl.annotate(text=f'SKNSH r: {s_100_all_pearson}',
                xy=(5.5,-1.5),
                fontsize=6)
    mpl.xlabel('Empirical l2FC')
    mpl.ylabel('Predicted l2FC')
    mpl.xlim(-2,10)
    mpl.ylim(-2,10)
    mpl.yticks(ticks=[0,5,10],labels=[0,5,10])
# hepg2 activity #
with sns.axes_style('white'):
    ax_02 = f.add_subplot(gs[1,0], sharex=ax_01)
    ax_02.set(adjustable='box', aspect='equal')
    sns.scatterplot(data=h_act_refactor_all,
                    x='empirical_l2fc',
                    y='predicted_l2fc',
                    color = '#FBB040',
                    s=1,
                    alpha=.1,
                    rasterized=True)
    mpl.plot([(.8 * min(h_act_refactor_all['empirical_l2fc'])), (.8 * max(h_act_refactor_all['empirical_l2fc']))],
             [(.8 * min(h_act_refactor_all['empirical_l2fc'])), (.8 * max(h_act_refactor_all['empirical_l2fc']))],
             linestyle='dashed',
             alpha=.5,
             color='k',
             linewidth=1)
    sns.kdeplot(data=h_act_refactor_all, x='empirical_l2fc', y='predicted_l2fc',color='#FBB040')
    mpl.annotate(text=f'HepG2 r: {h_100_all_pearson}',
                xy=(5.5,-1.5),
                fontsize=6)
    mpl.xlabel('')
    mpl.ylabel('Predicted l2FC')
    mpl.xlim(-2,10)
    mpl.ylim(-2,10)
    mpl.yticks(ticks=[0,5,10],labels=[0,5,10])
# k562 activity #
with sns.axes_style('white'):
    ax_03 = f.add_subplot(gs[0,0], sharex=ax_01)
    ax_03.set(adjustable='box', aspect='equal')
    mpl.plot([(.8 * min(k_act_refactor_all['empirical_l2fc'])), (.8 * max(k_act_refactor_all['empirical_l2fc']))],
             [(.8 * min(k_act_refactor_all['empirical_l2fc'])), (.8 * max(k_act_refactor_all['empirical_l2fc']))],
             linestyle='dashed',
             alpha=.5,
             color='k',
             linewidth=1)
    sns.scatterplot(data=k_act_refactor_all,
                    x='empirical_l2fc',
                    y='predicted_l2fc',
                    color = '#00A79D',
                    s=1,
                    alpha=.1,
                    rasterized=True)
    sns.kdeplot(data=k_act_refactor_all, x='empirical_l2fc', y='predicted_l2fc',color='#00A79D')
    mpl.annotate(text=f'K562 r: {k_100_all_pearson}',
                xy=(5.5,-1.5),
                fontsize=6)
    mpl.xlabel('')
    mpl.ylabel('Predicted l2FC')
    mpl.xlim(-2,10)
    mpl.ylim(-2,10)
    mpl.yticks(ticks=[0,5,10],labels=[0,5,10])
# sknsh skew
with sns.axes_style('white'):
    ax_04 = f.add_subplot(gs[2,1])
    ax_04.set(adjustable='box', aspect='equal')
    mpl.plot([(.8 * min(sknsh_all_emvar_avg['Log2Skew'])), (.8 * max(sknsh_all_emvar_avg['Log2Skew']))],
             [(.8 * min(sknsh_all_emvar_avg['Log2Skew'])), (.8 * max(sknsh_all_emvar_avg['Log2Skew']))],
             linestyle='dashed',
             alpha=.5,
             color='k',
             linewidth=1)
    sns.scatterplot(data=sknsh_all_emvar_avg,
                    x='Log2Skew',
                    y='sknsh_skew_pred_avg',
                    color = '#ED1C24',
                    s=3,
                    alpha=.1,
                    rasterized=True)
    sns.kdeplot(data=sknsh_all_emvar_avg, x='Log2Skew', y='sknsh_skew_pred_avg',color='#ED1C24')
    mpl.annotate(text=f'SKNSH r: {s_all_emvar_pearson}',
                xy=(1.5, -5.5),
                fontsize=6)
    mpl.xlabel('Empirical Skew')
    mpl.ylabel('Predicted Skew')
    mpl.xlim(-6, 6)
    mpl.ylim(-6, 6)
    mpl.yticks(ticks=[-5,0,5],labels=[-5,0,5])
# hepg2 skew #
with sns.axes_style('white'):
    ax_05 = f.add_subplot(gs[1,1])
    ax_05.set(adjustable='box', aspect='equal')
    mpl.plot([(.8 * min(hepg2_all_emvar_avg['Log2Skew'])), (.8 * max(hepg2_all_emvar_avg['Log2Skew']))],
             [(.8 * min(hepg2_all_emvar_avg['Log2Skew'])), (.8 * max(hepg2_all_emvar_avg['Log2Skew']))],
             linestyle='dashed',
             alpha=.5,
             color='k',
             linewidth=1)
    sns.scatterplot(data=hepg2_all_emvar_avg,
                    x='Log2Skew',
                    y='hepg2_skew_pred_avg',
                    color = '#FBB040',
                    s=3,
                    alpha=.1,
                    rasterized=True)
    sns.kdeplot(data=hepg2_all_emvar_avg, x='Log2Skew', y='hepg2_skew_pred_avg',color='#FBB040')
    mpl.annotate(text=f'HepG2 r: {h_all_emvar_pearson}',
                xy=(1.5, -5.5),
                fontsize=6)
    mpl.xlabel('')
    mpl.ylabel('Predicted Skew')
    mpl.xlim(-6, 6)
    mpl.ylim(-6, 6)
    mpl.yticks(ticks=[-5,0,5],labels=[-5,0,5])
# k562 skew #
with sns.axes_style('white'):
    ax_06 = f.add_subplot(gs[0,1])
    ax_06.set(adjustable='box', aspect='equal')
    mpl.plot([(.8 * min(k562_all_emvar_avg['Log2Skew'])), (.8 * max(k562_all_emvar_avg['Log2Skew']))],
             [(.8 * min(k562_all_emvar_avg['Log2Skew'])), (.8 * max(k562_all_emvar_avg['Log2Skew']))],
             linestyle='dashed',
             alpha=.5,
             color='k',
             linewidth=1)
    sns.scatterplot(data=k562_all_emvar_avg,
                    x='Log2Skew',
                    y='k562_skew_pred_avg',
                    color = '#00A79D',
                    s=3,
                    alpha=.1,
                    rasterized=True)
    sns.kdeplot(data=k562_all_emvar_avg, x='Log2Skew', y='k562_skew_pred_avg',color='#00A79D')
    mpl.annotate(text=f'K562 r: {k_all_emvar_pearson}',
                xy=(1.5, -5.5),
                fontsize=6)
    mpl.xlabel('')
    mpl.ylabel('Predicted Skew')
    mpl.xlim(-6, 6)
    mpl.ylim(-6, 6)
    mpl.yticks(ticks=[-5,0,5],labels=[-5,0,5])
mpl.tight_layout()
sns.despine()
mpl.tight_layout()
mpl.savefig('../analysis/fig1b_1c_scatterplots_with_indels_rev1_v1.pdf')

In [ ]:
# define function to plot scatters for activity and skew by chromosome
def chrom_specific_scatters_grid (merged_df, 
                      color, 
                      cell_type,
                      output_path):
    # get cell type
    if cell_type == 'K562':
        cell_lower = 'k562'
    elif cell_type == 'HEPG2':
        cell_lower = 'hepg2'
    elif cell_type == 'SKNSH':
        cell_lower = 'sknsh'
    # define grid position variables
    row = 0
    col = 0
    # filter for cell type
    cell_df = merged_df[merged_df['cell_type'] == cell_type]
    # get list of chromosomes
    autosomes = [f'chr{i}' for i in range(1,23)]
    correlation_dict = {}
    f = mpl.figure(figsize=(8,10),dpi=300)
    gs = f.add_gridspec(5,5)
    matplotlib.rcParams['pdf.fonttype'] = 42
    matplotlib.rcParams['ps.fonttype'] = 42
    for i in autosomes:
        # define x axis labels
        if row == 4:
            xlab = 'Empirical log2FC'
        elif row == 3 and col == 3:
            xlab = 'Empirical log2FC'
        elif row == 3 and col == 4:
            xlab = 'Empirical log2FC'
        elif row == 3 and col == 2:
            xlab = 'Empirical log2FC'
        else:
            xlab = ''
        # define y axis labels
        if col == 0:
            ylab = 'Predicted log2FC'
        else:
            ylab = ''
        # define ticks/labels
        if row == 4:
            xtick = [0,5,10]
        elif row == 3 and col == 3:
            xtick = [0,5,10]
        elif row == 3 and col == 4:
            xtick = [0,5,10]
        elif row == 3 and col == 2:
            xtick = [0,5,10]
        else:
            xtick = []
        if col == 0:
            ytick = [0,5,10]
        else:
            ytick = []
        chrom_df =cell_df[cell_df['chromosome'] == i]
        ref = chrom_df['A_log2FC'].tolist()
        alt = chrom_df['B_log2FC'].tolist()
        ref.extend(alt)
        # combine ref and alt preds for pearson r
        ref_pred = chrom_df[f'{cell_lower}_ref_pred_pos_100'].tolist()
        alt_pred = chrom_df[f'{cell_lower}_alt_pred_pos_100'].tolist()
        ref_pred.extend(alt_pred)
        # get pearson r
        pearson, pval = stats.pearsonr(x=ref, 
                                       y=ref_pred)
        round_pearson = round(pearson, 2)
        # plot KDE
        # make df for KDE
        # empirical data
        mpra_l2fc = chrom_df['A_log2FC'].tolist()
        mpra_l2fc.extend(chrom_df['B_log2FC'].tolist())
        # prediction data
        malinois_l2fc = chrom_df[f'{cell_lower}_ref_pred_pos_100'].tolist()
        malinois_l2fc.extend(chrom_df[f'{cell_lower}_alt_pred_pos_100'].tolist())
        kde_df = pd.DataFrame({'empirical_l2fc' : mpra_l2fc,
                               'malinois_l2fc' : malinois_l2fc})
        # plot correlation with KDE
        ax = f.add_subplot(gs[row,col])
        ax.set(adjustable='box', aspect='equal')
        mpl.plot([(.8 * min(chrom_df['A_log2FC'])), (.8 * max(chrom_df['A_log2FC']))],
                 [(.8 * min(chrom_df['A_log2FC'])), (.8 * max(chrom_df['A_log2FC']))],
                 linestyle='dashed',
                 alpha=.5,
                 color='k')
        sns.kdeplot(data=kde_df, x='empirical_l2fc', y='malinois_l2fc',color=color)
        # plot correlation
        sns.scatterplot(data=kde_df,
                        x='empirical_l2fc',
                        y='malinois_l2fc',
                        legend=False,
                        color=color,
                        s=3,
                        alpha=.1,
                        rasterized=True) 
        mpl.xlim(-2.5, 10)
        mpl.ylim(-2.5, 10)
        mpl.xlabel(xlab)
        mpl.ylabel(ylab)
        mpl.xticks(ticks=xtick, labels=xtick)
        mpl.yticks(ticks=ytick, labels=ytick)
        mpl.annotate(xy=(3,-1.5), text=f'{i} r: {round_pearson}', fontsize=8)
        correlation_dict.update({i : pearson})
        col +=1
        if col == 5:
            col=0
            row+=1
    sns.despine()
    mpl.tight_layout()
    #mpl.savefig(output_path)
    return correlation_dict

In [ ]:
# k562
k562_chrom_correlations  = chrom_specific_scatters_grid(snps_and_indels_merge, 
                                                   '#00A79D', 
                                                   'K562', 
                                                   '')

In [ ]:
# hepg2
hepg2_chrom_correlations  = chrom_specific_scatters_grid(snps_and_indels_merge, 
                                                         '#FBB040', 
                                                         'HEPG2', 
                                                         '')


In [ ]:
# sknsh
sknsh_chrom_correlations  = chrom_specific_scatters_grid(snps_and_indels_merge, 
                                                   '#ED1C24', 
                                                   'SKNSH', 
                                                   '')


In [ ]:
# define function to plot scatters for activity and skew by chromosome
def chrom_specific_scatters_with_bar (merged_df, 
                      color, 
                      cell_type,
                      output_path,
                      k_corr_dict,
                      h_corr_dict,
                      s_corr_dict):
    chrom_corr_bar_df = pd.concat([pd.DataFrame({'chrom' : [i for i in k_corr_dict.keys()],
                                             'corr' : [k_corr_dict.get(i) for i in k_corr_dict.keys()],
                                             'cell_type' : ['K562' for i in range(len(k_corr_dict.keys()))]}),
                               pd.DataFrame({'chrom' : [i for i in h_corr_dict.keys()],
                                             'corr' : [h_corr_dict.get(i) for i in h_corr_dict.keys()],
                                             'cell_type' : ['HEPG2' for i in range(len(h_corr_dict.keys()))]}),
                               pd.DataFrame({'chrom' : [i for i in s_corr_dict.keys()],
                                             'corr' : [s_corr_dict.get(i) for i in s_corr_dict.keys()],
                                             'cell_type' : ['SKNSH' for i in range(len(s_corr_dict.keys()))]})])
    # make a palette to color the bars
    bar_palette = {'K562' : '#00A79D',
               'HEPG2' : '#FBB040',
               'SKNSH' : '#ED1C24'}
    # get cell type
    if cell_type == 'K562':
        cell_lower = 'k562'
    elif cell_type == 'HEPG2':
        cell_lower = 'hepg2'
    elif cell_type == 'SKNSH':
        cell_lower = 'sknsh'
    # define grid position variables
    row = 0
    col = 0
    # filter for cell type
    cell_df = merged_df[merged_df['cell_type'] == cell_type]
    # get list of chromosomes
    autosomes = [f'chr{i}' for i in range(1,23)]
    correlation_dict = {}
    f = mpl.figure(figsize=(8,10),dpi=300)
    gs = f.add_gridspec(5,5)
    matplotlib.rcParams['pdf.fonttype'] = 42
    matplotlib.rcParams['ps.fonttype'] = 42
    for i in autosomes:
        # define x axis labels
        if row == 4:
            xlab = 'Empirical log2FC'
        elif row == 3 and col == 3:
            xlab = 'Empirical log2FC'
        elif row == 3 and col == 4:
            xlab = 'Empirical log2FC'
        elif row == 3 and col == 2:
            xlab = 'Empirical log2FC'
        else:
            xlab = ''
        # define y axis labels
        if col == 0:
            ylab = 'Predicted log2FC'
        else:
            ylab = ''
        # define ticks/labels
        if row == 4:
            xtick = [0,5,10]
        elif row == 3 and col == 3:
            xtick = [0,5,10]
        elif row == 3 and col == 4:
            xtick = [0,5,10]
        elif row == 3 and col == 2:
            xtick = [0,5,10]
        else:
            xtick = []
        if col == 0:
            ytick = [0,5,10]
        else:
            ytick = []
        chrom_df =cell_df[cell_df['chromosome'] == i]
        ref = chrom_df['A_log2FC'].tolist()
        alt = chrom_df['B_log2FC'].tolist()
        ref.extend(alt)
        # combine ref and alt preds for pearson r
        ref_pred = chrom_df[f'{cell_lower}_ref_pred_pos_100'].tolist()
        alt_pred = chrom_df[f'{cell_lower}_alt_pred_pos_100'].tolist()
        ref_pred.extend(alt_pred)
        # get pearson r
        pearson, pval = stats.pearsonr(x=ref, 
                                       y=ref_pred)
        round_pearson = round(pearson, 2)
        # plot KDE
        # make df for KDE
        # empirical data
        mpra_l2fc = chrom_df['A_log2FC'].tolist()
        mpra_l2fc.extend(chrom_df['B_log2FC'].tolist())
        # prediction data
        malinois_l2fc = chrom_df[f'{cell_lower}_ref_pred_pos_100'].tolist()
        malinois_l2fc.extend(chrom_df[f'{cell_lower}_alt_pred_pos_100'].tolist())
        kde_df = pd.DataFrame({'empirical_l2fc' : mpra_l2fc,
                               'malinois_l2fc' : malinois_l2fc})
        # plot correlation with KDE
        ax = f.add_subplot(gs[row,col])
        #ax.set(adjustable='box', aspect='equal')
        mpl.plot([(.8 * min(chrom_df['A_log2FC'])), (.8 * max(chrom_df['A_log2FC']))],
                 [(.8 * min(chrom_df['A_log2FC'])), (.8 * max(chrom_df['A_log2FC']))],
                 linestyle='dashed',
                 alpha=.5,
                 color='k')
        sns.kdeplot(data=kde_df, x='empirical_l2fc', y='malinois_l2fc',color=color)
        # plot correlation
        sns.scatterplot(data=kde_df,
                        x='empirical_l2fc',
                        y='malinois_l2fc',
                        legend=False,
                        color=color,
                        s=3,
                        alpha=.1,
                        rasterized=True) 
        mpl.xlim(-2.5, 10)
        mpl.ylim(-2.5, 10)
        mpl.xlabel(xlab)
        mpl.ylabel(ylab)
        mpl.xticks(ticks=xtick, labels=xtick)
        mpl.yticks(ticks=ytick, labels=ytick)
        mpl.annotate(xy=(3,-1.5), text=f'{i} r: {round_pearson}', fontsize=8)
        correlation_dict.update({i : pearson})
        col +=1
        if col == 5:
            col=0
            row+=1
    ax = f.add_subplot(gs[4,2:])
    sns.barplot(data=chrom_corr_bar_df,
            x='chrom',
            y='corr',
            hue='cell_type',
            palette=bar_palette)
    mpl.ylim(.80, 0.95)
    #mpl.yticks([0.80, 0.82, 0.84, 0.86, 0.88, 0.90, 0.92],
    #           [0.80, 0.82, 0.84, 0.86, 0.88, 0.90, 0.92])
    mpl.ylabel('')
    mpl.xlabel('')
    mpl.legend(loc='upper right', fontsize='x-small')
    mpl.xticks(rotation=90)
    sns.despine()
    mpl.tight_layout()
    mpl.savefig(output_path)
    return correlation_dict

In [ ]:
# k562
k562_chrom_correlations = chrom_specific_scatters_with_bar(snps_and_indels_merge, 
                                                   '#00A79D', 
                                                   'K562', 
                                                   '../analysis/k562_activity_corr_by_chrom_with_indels_suppfig1b_rev1_v2.pdf',
                                                   k562_chrom_correlations,
                                                   hepg2_chrom_correlations,
                                                   sknsh_chrom_correlations)

In [ ]:
# iterate through each chromosome and print the cell type of the highest correlation as well the difference between the max and sknsh for that cell type
# make a list for storing delta sknsh
deltaSKNSH = []
lessHundreth = []
for chrom in [f'chr{i}' for i in range(1,23)]:
    print(f'### {chrom} ###')
    # get correlation for k562
    k_corr = k562_chrom_correlations.get(chrom)
    # get correlation for hepg2
    h_corr = hepg2_chrom_correlations.get(chrom)
    # get correlation for sknsh
    s_corr = sknsh_chrom_correlations.get(chrom)
    # get the max
    max_corr = np.max([k_corr, h_corr, s_corr])
    if max_corr == k_corr:
        deltaS = k_corr - s_corr
        print(f'k562 has the highest correlation, delta sknsh is {deltaS}')
        deltaSKNSH.append({deltaS})
        if deltaS < float(0.01):
            lessHundreth.append(1)
    elif max_corr == h_corr:
        deltaS = h_corr - s_corr
        print(f'hepg2 has the highest correlation, delta sknsh is {deltaS}')
        deltaSKNSH.append({deltaS})
        if deltaS < float(0.01):
            lessHundreth.append(1)
    elif max_corr == s_corr:
        print(f'sknsh has the highest correlation for {chrom},  delta sknsh is {s_corr - s_corr}')
print(f'the greatest delta SKNSH correlation is {np.max(deltaSKNSH)}')
print(f'There are {np.sum(lessHundreth)} chromosomes where the SKSH difference is < 0.01')

In [ ]:
# refactor activities for plotting in grid
# center pred #
k_indel_refactor = refactor_activities('K562', raw_less_ten, 'pos_100')
h_indel_refactor = refactor_activities('HEPG2', raw_less_ten, 'pos_100')
s_indel_refactor = refactor_activities('SKNSH', raw_less_ten, 'pos_100')

In [ ]:
### get correlations for annotating plots ###
### position 100 prediction activity ###
# k562
k_100_indel_pearson = round(stats.pearsonr(x=k_indel_refactor['empirical_l2fc'],
                                           y=k_indel_refactor['predicted_l2fc'])[0], 2)
# hepg2
h_100_indel_pearson = round(stats.pearsonr(x=h_indel_refactor['empirical_l2fc'],
                                           y=h_indel_refactor['predicted_l2fc'])[0] ,2)
# sknsh
s_100_indel_pearson = round(stats.pearsonr(x=s_indel_refactor['empirical_l2fc'],
                                           y=s_indel_refactor['predicted_l2fc'])[0], 2)

In [ ]:
# plot activity correlations
f = mpl.figure(figsize=(8.5,3),dpi=300)
gs = f.add_gridspec(1,3)
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
### plot activities in column 1 ###
# sknsh activity #
with sns.axes_style():
    ax_01 = f.add_subplot(gs[0,2])
    ax_01.set_aspect(1)
    mpl.plot([(.8 * min(s_indel_refactor['empirical_l2fc'])), (.8 * max(s_indel_refactor['empirical_l2fc']))],
             [(.8 * min(s_indel_refactor['empirical_l2fc'])), (.8 * max(s_indel_refactor['empirical_l2fc']))],
             linestyle='dashed',
             alpha=.5,
             color='k',
             linewidth=1)
    sns.scatterplot(data=s_indel_refactor,
                    x='empirical_l2fc',
                    y='predicted_l2fc',
                    color = '#ED1C24',
                    s=1,
                    alpha=.25,
                    rasterized=True)
    sns.kdeplot(data=s_indel_refactor, x='empirical_l2fc', y='predicted_l2fc',color='#ED1C24', linewidths=.5)
    mpl.annotate(text=f'SKNSH r: {s_100_indel_pearson}',
                xy=(7,-0.5),
                fontsize=6)
    mpl.annotate(text=f'n: {len(s_indel_refactor)}',
                 xy=(7,-1.5),
                 fontsize=6)
    mpl.xlabel('Empirical l2FC', fontsize=8)
    mpl.ylabel('')
    mpl.xlim(-2,10)
    mpl.ylim(-2,10)
    mpl.xticks(ticks=[0,5,10],labels=[0,5,10], fontsize=6)
    mpl.yticks(ticks=[0,5,10],labels=[0,5,10], fontsize=6)
    
# hepg2 activity #
with sns.axes_style():
    ax_02 = f.add_subplot(gs[0,1], sharex=ax_01)
    ax_02.set_aspect(1)
    sns.scatterplot(data=h_indel_refactor,
                    x='empirical_l2fc',
                    y='predicted_l2fc',
                    color = '#FBB040',
                    s=1,
                    alpha=.5,
                    rasterized=True)
    mpl.plot([(.8 * min(h_indel_refactor['empirical_l2fc'])), (.8 * max(h_indel_refactor['empirical_l2fc']))],
             [(.8 * min(h_indel_refactor['empirical_l2fc'])), (.8 * max(h_indel_refactor['empirical_l2fc']))],
             linestyle='dashed',
             alpha=.25,
             color='k',
             linewidth=1)
    sns.kdeplot(data=h_indel_refactor, x='empirical_l2fc', y='predicted_l2fc',color='#FBB040', linewidths=.5)
    mpl.annotate(text=f'HepG2 r: {h_100_indel_pearson}',
                xy=(7,-0.5),
                fontsize=6)
    mpl.annotate(text=f'n: {len(h_indel_refactor)}',
                 xy=(7,-1.5),
                 fontsize=6)
    mpl.xlabel('Empirical l2FC', fontsize=8)
    mpl.ylabel('')
    mpl.xlim(-2,10)
    mpl.ylim(-2,10)
    mpl.xticks(ticks=[0,5,10],labels=[0,5,10], fontsize=6)
    mpl.yticks(ticks=[0,5,10],labels=[0,5,10], fontsize=6)
    
# k562 activity #
with sns.axes_style():
    ax_03 = f.add_subplot(gs[0,0], sharex=ax_01)
    ax_03.set_aspect(1)
    mpl.plot([(.8 * min(k_indel_refactor['empirical_l2fc'])), (.8 * max(k_indel_refactor['empirical_l2fc']))],
             [(.8 * min(k_indel_refactor['empirical_l2fc'])), (.8 * max(k_indel_refactor['empirical_l2fc']))],
             linestyle='dashed',
             alpha=.5,
             color='k',
             linewidth=1)
    sns.scatterplot(data=k_indel_refactor,
                    x='empirical_l2fc',
                    y='predicted_l2fc',
                    color = '#00A79D',
                    s=1,
                    alpha=.25,
                    rasterized=True)
    sns.kdeplot(data=k_indel_refactor, x='empirical_l2fc', y='predicted_l2fc',color='#00A79D', linewidths=.5)
    mpl.annotate(text=f'K562 r: {k_100_indel_pearson}',
                xy=(7,-0.5),
                fontsize=6)
    mpl.annotate(text=f'n: {len(k_indel_refactor)}',
                 xy=(7,-1.5),
                 fontsize=6)
    mpl.xlabel('Empirical l2FC', fontsize=8)
    mpl.ylabel('Predicted l2FC', fontsize=8)
    mpl.xlim(-2,10)
    mpl.ylim(-2,10)
    mpl.xticks(ticks=[0,5,10],labels=[0,5,10], fontsize=6)
    mpl.yticks(ticks=[0,5,10],labels=[0,5,10], fontsize=6)
    
sns.despine()
mpl.tight_layout()
#mpl.savefig('../analysis/suppfig3a_scatterplots_with_indels_rev1_v1.pdf')

In [ ]:
# re-plot correlations but color points by size of the indel #
# calculate length of indel #
# activity #
k_indel_refactor.loc[:,'len_indel'] = [abs(len(var.split(':')[-2]) - len(var.split(':')[-1])) for var in k_indel_refactor['variant']]
h_indel_refactor.loc[:,'len_indel'] = [abs(len(var.split(':')[-2]) - len(var.split(':')[-1])) for var in h_indel_refactor['variant']]
s_indel_refactor.loc[:,'len_indel'] = [abs(len(var.split(':')[-2]) - len(var.split(':')[-1])) for var in s_indel_refactor['variant']]

In [ ]:
# define function for plotting by indel size #
def act_corr_by_indel_length(refactor_df,
                             cell_color,
                             cell_type,
                             out_name):
    # get length of indels #
    indel_lengths = sorted(list(refactor_df['len_indel'].unique()))
    # set up plot #
    f = mpl.figure(dpi=300, figsize=(8.5,5.5))
    gs = f.add_gridspec(3,3)
    matplotlib.rcParams['pdf.fonttype'] = 42
    matplotlib.rcParams['ps.fonttype'] = 42
    # iterate through indel lengths and plot #
    # make counter for adding coordinates to grid #
    col_count = 0
    row_count = 0
    for indel_size in indel_lengths:
        # filter for variants matching the indel length #
        indel_df = refactor_df[refactor_df['len_indel'] == indel_size]
        # get correlation #
        indel_corr = round(stats.pearsonr(x=indel_df['empirical_l2fc'],
                                          y=indel_df['predicted_l2fc'])[0], 2)
        # plot #
        ax = f.add_subplot(gs[row_count,
                              col_count])
        ax.set_aspect(1)
        sns.scatterplot(data=indel_df,
                    x='empirical_l2fc',
                    y='predicted_l2fc',
                    color = cell_color,
                    s=1,
                    alpha=.25,
                    rasterized=True)
        sns.kdeplot(data=indel_df, x='empirical_l2fc', y='predicted_l2fc',color=cell_color, linewidths=.25)
        mpl.annotate(text=f'{cell_type} r: {indel_corr}',
                xy=(5.5,-1.5),
                fontsize=6)
        if row_count == 2 and col_count == 0:
            mpl.xlabel('Empirical l2FC', fontsize=8)
            mpl.ylabel('Predicted l2FC', fontsize=8)
            mpl.xlim(-2,10)
            mpl.ylim(-2,10)
            mpl.yticks(ticks=[0,5,10],labels=[0,5,10], fontsize=6)
            mpl.xticks(ticks=[0,5,10],labels=[0,5,10], fontsize=6)
        elif col_count == 0 and row_count != 5:
            mpl.xlabel('')
            mpl.ylabel('Predicted l2FC', fontsize=8)
            mpl.xlim(-2,10)
            mpl.ylim(-2,10)
            mpl.xticks(ticks=[0,5,10],labels=['', '', ''], fontsize=6)
            mpl.yticks(ticks=[0,5,10],labels=[0,5,10], fontsize=6)
        elif row_count == 2 and col_count > 0:
            mpl.xlabel('Empirical l2FC', fontsize=8)
            mpl.ylabel('')
            mpl.xlim(-2,10)
            mpl.ylim(-2,10)
            mpl.xticks(ticks=[0,5,10],labels=[0,5,10], fontsize=6)
            mpl.yticks(ticks=[0,5,10],labels=['', '', ''], fontsize=6)
        else:
            mpl.xlabel('')
            mpl.ylabel('')
            mpl.xlim(-2,10)
            mpl.ylim(-2,10)
            mpl.xticks(ticks=[0,5,10],labels=['', '', ''], fontsize=6)
            mpl.yticks(ticks=[0,5,10],labels=['', '', ''], fontsize=6)
        mpl.title(f'len(indel): {indel_size} | n:{len(indel_df)}', loc='left', fontsize=6)
        sns.despine()
        mpl.tight_layout()
        mpl.savefig(out_name)
        # update column counter #
        if col_count == 2:
            col_count = 0
            row_count +=1
        else:
            col_count +=1 

In [ ]:
# print number of indels tested across the three cell types
print(len(pd.concat([k_indel_refactor, h_indel_refactor, s_indel_refactor])['hg38_id'].unique()))

In [ ]:
# plot activity correlation by indel length #
# K562 #
act_corr_by_indel_length(k_indel_refactor, 
                         '#00A79D', 
                         'K562', 
                         '../analysis/k562_activity_correlation_by_indel_length.pdf')

In [ ]:
# plot activity correlation by indel length #
# HepG2 #
act_corr_by_indel_length(h_indel_refactor, 
                         '#FBB040', 
                         'HepG2', 
                         '../analysis/hepg2_activity_correlation_by_indel_length.pdf')

In [ ]:
# plot activity correlation by indel length #
# SKNSH #
act_corr_by_indel_length(s_indel_refactor, 
                         '#ED1C24', 
                         'SKNSH', 
                         '../analysis/sknsh_activity_correlation_by_indel_length.pdf')

In [ ]:
# define function to plot skew correlations for emVars
def emvar_skew_scatter (merged_df, 
                        cell_type):
    # get cell type
    if cell_type == 'K562':
        cell_lower = 'k562'
    elif cell_type == 'HEPG2':
        cell_lower = 'hepg2'
    elif cell_type == 'SKNSH':
        cell_lower = 'sknsh'
    # filter for given cell t
    # filter for cell type
    cell_df = merged_df[(merged_df['cell_type'] == cell_type) &
                        (merged_df['emVar'] == True)].copy()
    # get pearson r
    pearson, pval = stats.pearsonr(x=cell_df['Log2Skew'], 
                                   y=cell_df[f'{cell_lower}_skew_pred_avg'])
    print(pearson)
    print(pval)
    print(len(cell_df))
    pearson = round(pearson, 2)
    return cell_df

In [ ]:
# filter full data for only emVars for plotting skew
# average of preds
k562_emvar_indel_avg = emvar_skew_scatter(raw_less_ten, 'K562')
hepg2_emvar_indel_avg = emvar_skew_scatter(raw_less_ten, 'HEPG2')
sknsh_emvar_indel_avg = emvar_skew_scatter(raw_less_ten, 'SKNSH')

In [ ]:
### get correlations for annotating plots ###
### emVars ###
# k562
k_indel_emvar_pearson = round(stats.pearsonr(k562_emvar_indel_avg['Log2Skew'],
                                             k562_emvar_indel_avg['k562_skew_pred_avg'])[0], 2)
# hepg2
h_indel_emvar_pearson = round(stats.pearsonr(hepg2_emvar_indel_avg['Log2Skew'],
                                           hepg2_emvar_indel_avg['hepg2_skew_pred_avg'])[0], 2)
# sknsh
s_indel_emvar_pearson = round(stats.pearsonr(sknsh_emvar_indel_avg['Log2Skew'],
                                             sknsh_emvar_indel_avg['sknsh_skew_pred_avg'])[0], 2)

In [ ]:
# plot skews
f = mpl.figure(figsize=(8.5,3),dpi=300)
gs = f.add_gridspec(1,3)
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
# sknsh skew
with sns.axes_style():
    ax_04 = f.add_subplot(gs[0,2])
    ax_04.set_aspect(1)
    sns.scatterplot(data=sknsh_emvar_indel_avg,
                    x='Log2Skew',
                    y='sknsh_skew_pred_avg',
                    color = '#ED1C24',
                    s=5,
                    alpha=.35,
                    rasterized=True)
    sns.kdeplot(data=sknsh_emvar_indel_avg, x='Log2Skew', y='sknsh_skew_pred_avg',color='#ED1C24', linewidths=1, alpha=.5)
    mpl.annotate(text=f'SKNSH r: {s_indel_emvar_pearson}',
                xy=(2.5, -4.5),
                fontsize=6)
    mpl.annotate(text=f'n: {len(sknsh_emvar_indel_avg)}',
                xy=(2.5, -5.5),
                fontsize=6)
    mpl.xlabel('Empirical Skew', fontsize=8)
    mpl.ylabel('')
    mpl.xlim(-6, 6)
    mpl.ylim(-6, 6)
    mpl.yticks(ticks=[-5,0,5],labels=[-5,0,5], fontsize=6)
    mpl.xticks(ticks=[-5,0,5],labels=[-5,0,5], fontsize=6)
# hepg2 skew #
with sns.axes_style():
    ax_05 = f.add_subplot(gs[0,1])
    ax_05.set_aspect(1)
    sns.scatterplot(data=hepg2_emvar_indel_avg,
                    x='Log2Skew',
                    y='hepg2_skew_pred_avg',
                    color = '#FBB040',
                    s=5,
                    alpha=.35,
                    rasterized=True)
    sns.kdeplot(data=hepg2_emvar_indel_avg, x='Log2Skew', y='hepg2_skew_pred_avg',color='#FBB040', linewidths=1, alpha=.5)
    mpl.annotate(text=f'HepG2 r: {h_indel_emvar_pearson}',
                xy=(2.5, -4.5),
                fontsize=6)
    mpl.annotate(text=f'n: {len(hepg2_emvar_indel_avg)}',
                xy=(2.5, -5.5),
                fontsize=6)
    mpl.xlabel('Empirical Skew', fontsize=8)
    mpl.ylabel('')
    mpl.xlim(-6, 6)
    mpl.ylim(-6, 6)
    mpl.yticks(ticks=[-5,0,5],labels=[-5,0,5], fontsize=6)
    mpl.xticks(ticks=[-5,0,5],labels=[-5,0,5], fontsize=6)
# k562 skew #
with sns.axes_style():
    ax_06 = f.add_subplot(gs[0,0])
    ax_06.set(adjustable='box', aspect='equal')
    sns.scatterplot(data=k562_emvar_indel_avg,
                    x='Log2Skew',
                    y='k562_skew_pred_avg',
                    color = '#00A79D',
                    s=5,
                    alpha=.35,
                    rasterized=True)
    sns.kdeplot(data=hepg2_emvar_indel_avg, x='Log2Skew', y='k562_skew_pred_avg',color='#00A79D', linewidths=1, alpha=.5)
    mpl.annotate(text=f'K562 r: {k_indel_emvar_pearson}',
                xy=(2.5, -4.5),
                fontsize=6)
    mpl.annotate(text=f'n: {len(k562_emvar_indel_avg)}',
                xy=(2.5, -5.5),
                fontsize=6)
    mpl.xlabel('Empirical Skew', fontsize=8)
    mpl.ylabel('Predicted Skew', fontsize=8)
    mpl.xlim(-6, 6)
    mpl.ylim(-6, 6)
    mpl.yticks(ticks=[-5,0,5],labels=[-5,0,5], fontsize=6)
    mpl.xticks(ticks=[-5,0,5],labels=[-5,0,5], fontsize=6)
sns.despine()
mpl.tight_layout()
mpl.savefig('../analysis/emVar_indel_correlation.pdf')

In [ ]:
# add indel length for emVars
k562_emvar_indel_avg.loc[:,'len_indel'] = [abs(len(var.split(':')[-2]) - len(var.split(':')[-1])) for var in k562_emvar_indel_avg['variant']]
hepg2_emvar_indel_avg.loc[:,'len_indel'] = [abs(len(var.split(':')[-2]) - len(var.split(':')[-1])) for var in hepg2_emvar_indel_avg['variant']]
sknsh_emvar_indel_avg.loc[:,'len_indel'] = [abs(len(var.split(':')[-2]) - len(var.split(':')[-1])) for var in sknsh_emvar_indel_avg['variant']]
# add label for skew direction #
k562_emvar_indel_avg.loc[:,'ins_del'] = ['Deletion' if (len(var.split(':')[-2]) - len(var.split(':')[-1])) >= 0 else 'Insertion' for var in k562_emvar_indel_avg['variant']]
hepg2_emvar_indel_avg.loc[:,'ins_del'] = ['Deletion' if (len(var.split(':')[-2]) - len(var.split(':')[-1])) >= 0 else 'Insertion' for var in hepg2_emvar_indel_avg['variant']]
sknsh_emvar_indel_avg.loc[:,'ins_del'] = ['Deletion' if (len(var.split(':')[-2]) - len(var.split(':')[-1])) >= 0 else 'Insertion' for var in sknsh_emvar_indel_avg['variant']]

In [ ]:
# get total number of emVars across the three cell types
print(len(pd.concat([k562_emvar_indel_avg, hepg2_emvar_indel_avg, sknsh_emvar_indel_avg])['variant'].unique()))

In [ ]:
# define function for plotting emvars by indel size #
def skew_corr_by_indel_length_abs(emvar_df,
                                     cell_color,
                                     cell_type,
                                     out_name):
    # get length of indels #
    indel_lengths = sorted(list(emvar_df['len_indel'].unique()))
    
    # Filter out indel lengths with <= 100 variants
    valid_indel_lengths = []
    for indel_size in indel_lengths:
        indel_df = emvar_df[emvar_df['len_indel'] == indel_size]
        if len(indel_df) > 100:
            valid_indel_lengths.append(indel_size)
    
    # If no valid bins, return early
    if len(valid_indel_lengths) == 0:
        print("No indel bins with > 100 variants")
        return
    
    # Calculate grid dimensions based on number of valid bins
    n_plots = len(valid_indel_lengths)
    n_cols = 4
    n_rows = int(np.ceil(n_plots / n_cols))
    
    # Determine which plots are the last in each column
    last_in_column = {}
    for i, indel_size in enumerate(valid_indel_lengths):
        col = i % n_cols
        last_in_column[col] = i  # Keep updating, last one will be the final plot in that column
    
    # set up plot #
    f = mpl.figure(dpi=300, figsize=(8.5, 5))
    gs = f.add_gridspec(n_rows, n_cols, hspace=0.15, wspace=0.15)
    matplotlib.rcParams['pdf.fonttype'] = 42
    matplotlib.rcParams['ps.fonttype'] = 42
    
    # iterate through valid indel lengths and plot #
    for plot_idx, indel_size in enumerate(valid_indel_lengths):
        col_count = plot_idx % n_cols
        row_count = plot_idx // n_cols
        
        # filter for variants matching the indel length #
        indel_df = emvar_df[emvar_df['len_indel'] == indel_size]
        
        # get correlation #
        indel_corr = round(stats.pearsonr(x=indel_df['Log2Skew'],
                                          y=indel_df[f'{cell_type.lower()}_skew_pred_avg'])[0], 2)
        # plot #
        ax = f.add_subplot(gs[row_count, col_count])
        ax.set_aspect(1)
        
        sns.scatterplot(data=indel_df,
                    x='Log2Skew',
                    y=f'{cell_type.lower()}_skew_pred_avg',
                    color = cell_color,
                    s=5,
                    alpha=.75,
                    rasterized=True)
        sns.kdeplot(data=indel_df, x='Log2Skew', y=f'{cell_type.lower()}_skew_pred_avg',
                   color=cell_color, linewidths=1, alpha=.5)
        mpl.annotate(text=f'{cell_type} r: {indel_corr}',
                xy=(.5,-5.5),
                fontsize=6)
        
        # Dynamic axis labeling based on position
        # Last in column: show x-axis labels
        is_last_in_column = (plot_idx == last_in_column[col_count])
        # Left column: show y-axis labels
        is_left_column = (col_count == 0)
        
        if is_last_in_column and is_left_column:
            mpl.xlabel('Empirical Skew', fontsize=10)
            mpl.ylabel('Predicted Skew', fontsize=10)
            mpl.xlim(-6,6)
            mpl.ylim(-6,6)
            mpl.yticks(ticks=[-4,0,4],labels=[-4,0,4], fontsize=8)
            mpl.xticks(ticks=[-4,0,4],labels=[-4,0,4], fontsize=8)
        elif is_left_column:
            mpl.xlabel('')
            mpl.ylabel('Predicted Skew', fontsize=10)
            mpl.xlim(-6,6)
            mpl.ylim(-6,6)
            mpl.xticks(ticks=[-4,0,4],labels=['', '', ''])
            mpl.yticks(ticks=[-4,0,4],labels=[-4,0,4], fontsize=8)
        elif is_last_in_column:
            mpl.xlabel('Empirical Skew', fontsize=10)
            mpl.ylabel('')
            mpl.xlim(-6,6)
            mpl.ylim(-6,6)
            mpl.xticks(ticks=[-4,0,4],labels=[-4,0,4], fontsize=8)
            mpl.yticks(ticks=[-4,0,4],labels=['', '', ''], fontsize=8)
        else:
            mpl.xlabel('')
            mpl.ylabel('')
            mpl.xlim(-6, 6)
            mpl.ylim(-6, 6)
            mpl.xticks(ticks=[-4,0,4],labels=['', '', ''])
            mpl.yticks(ticks=[-4,0,4],labels=['', '', ''])
        
        mpl.title(f'len(indel): {indel_size} | n:{len(indel_df)}', loc='left', fontsize=8)
        sns.despine()
    
    # Adjust layout
    f.subplots_adjust(left=0.08, right=0.98, top=0.95, bottom=0.12)
    mpl.savefig(out_name)

In [ ]:
# plot correlation of emvars by indel length #
# K562 #
skew_corr_by_indel_length_abs(k562_emvar_indel_avg, 
                              '#00A79D', 
                              'k562', 
                              '../analysis/k562_emvar_skew_correlation_by_indel_length_v2.pdf')

In [ ]:
# hepg2 #
skew_corr_by_indel_length_abs(hepg2_emvar_indel_avg, 
                              '#FBB040', 
                              'hepg2',
                              '../analysis/hepg2_emvar_skew_correlation_by_indel_length.pdf')

In [ ]:
# sknsh #
skew_corr_by_indel_length_abs(sknsh_emvar_indel_avg, 
                              '#ED1C24', 
                              'sknsh',
                              '../analysis/sknsh_emvar_skew_correlation_by_indel_length.pdf')

In [ ]:
# define function for plotting emvars by indel size #
def skew_corr_by_indel_length_signed(emvar_df,
                                     cell_color,
                                     cell_type,
                                     out_name):
    # create palette for coloring plots #
    indel_pal = {'Insertion' : 'orange',
                 'Deletion' : '#253494'}
    # make labels #
    ins_patch = mpatches.Patch(color='orange', label='Insertion')
    del_patch = mpatches.Patch(color='#253494', label='Deletion')
    
    # get length of indels #
    indel_lengths = sorted(list(emvar_df['len_indel'].unique()))
    
    # Filter out indel lengths with <= 100 variants
    valid_indel_lengths = []
    for indel_size in indel_lengths:
        indel_df = emvar_df[emvar_df['len_indel'] == indel_size]
        if len(indel_df) > 100:
            valid_indel_lengths.append(indel_size)
    
    # If no valid bins, return early
    if len(valid_indel_lengths) == 0:
        print("No indel bins with > 100 variants")
        return
    
    # Calculate grid dimensions based on number of valid bins
    n_plots = len(valid_indel_lengths)
    n_cols = 3
    n_rows = int(np.ceil(n_plots / n_cols))
    
    # Determine which plots are the last in each column
    last_in_column = {}
    for i in range(n_plots):
        col = i % n_cols
        last_in_column[col] = i
    
    # set up plot #
    f = mpl.figure(dpi=300, figsize=(8.5, 6))
    gs = f.add_gridspec(n_rows, n_cols)
    matplotlib.rcParams['pdf.fonttype'] = 42
    matplotlib.rcParams['ps.fonttype'] = 42
    
    # iterate through valid indel lengths and plot #
    for plot_idx, indel_size in enumerate(valid_indel_lengths):
        col_count = plot_idx % n_cols
        row_count = plot_idx // n_cols
        
        # filter for variants matching the indel length #
        indel_df = emvar_df[emvar_df['len_indel'] == indel_size]
        
        # get correlation #
        indel_corr = round(stats.pearsonr(x=indel_df['Log2Skew'],
                                          y=indel_df[f'{cell_type.lower()}_skew_pred_avg'])[0], 2)
        # plot #
        ax = f.add_subplot(gs[row_count, col_count])
        ax.set_aspect(1)
        sns.scatterplot(data=indel_df,
                    x='Log2Skew',
                    y=f'{cell_type.lower()}_skew_pred_avg',
                    hue = 'ins_del',
                    s=5,
                    alpha=.75,
                    rasterized=True,
                    legend=False,
                    palette=indel_pal)
        mpl.legend(handles=[ins_patch, del_patch],
                   loc='upper left',
                   fontsize=4,
                   frameon=False)
        sns.kdeplot(data=indel_df, x='Log2Skew', y=f'{cell_type.lower()}_skew_pred_avg',
                   color='lightslategrey', alpha=.25, linewidths=.5)
        mpl.annotate(text=f'{cell_type} r: {indel_corr}',
                xy=(.5,-5.5),
                fontsize=6)
        
        # Dynamic axis labeling based on position
        # Last in column: show x-axis labels
        is_last_in_column = (plot_idx == last_in_column[col_count])
        # Left column: show y-axis labels
        is_left_column = (col_count == 0)
        
        if is_last_in_column and is_left_column:
            mpl.xlabel('Empirical Skew', fontsize=10)
            mpl.ylabel('Predicted Skew', fontsize=10)
            mpl.xlim(-6,6)
            mpl.ylim(-6,6)
            mpl.yticks(ticks=[-4,0,4],labels=[-4,0,4], fontsize=8)
            mpl.xticks(ticks=[-4,0,4],labels=[-4,0,4], fontsize=8)
        elif is_left_column:
            mpl.xlabel('')
            mpl.ylabel('Predicted Skew', fontsize=10)
            mpl.xlim(-6,6)
            mpl.ylim(-6,6)
            mpl.xticks(ticks=[-4,0,4],labels=['', '', ''])
            mpl.yticks(ticks=[-4,0,4],labels=[-4,0,4], fontsize=8)
        elif is_last_in_column:
            mpl.xlabel('Empirical Skew', fontsize=10)
            mpl.ylabel('')
            mpl.xlim(-6,6)
            mpl.ylim(-6,6)
            mpl.xticks(ticks=[-4,0,4],labels=[-4,0,4], fontsize=8)
            mpl.yticks(ticks=[-4,0,4],labels=['', '', ''])
        else:
            mpl.xlabel('')
            mpl.ylabel('')
            mpl.xlim(-6, 6)
            mpl.ylim(-6, 6)
            mpl.xticks(ticks=[-4,0,4],labels=['', '', ''])
            mpl.yticks(ticks=[-4,0,4],labels=['', '', ''])
        
        mpl.title(f'len(indel): {indel_size} | n:{len(indel_df)}', loc='left', fontsize=6)
        sns.despine()
    
    # Adjust layout and save (moved outside loop)
    f.subplots_adjust(left=0.08, right=0.98, top=0.95, bottom=0.10)
    mpl.savefig(out_name)

In [ ]:
# plot scatterplots again but color by insertion or deletion #
# k562 #
skew_corr_by_indel_length_signed(k562_emvar_indel_avg, 
                              '#00A79D', 
                              'k562', 
                              '../analysis/k562_emvar_skew_correlation_by_indel_length_INDEL_COLORED.pdf')

In [ ]:
len(k562_emvar_indel_avg)

In [ ]:
k562_emvar_indel_avg['id']

In [ ]:
# plot differences in skew distribution between SNVs and Indels
# filter for SNV emVars
k562_emvar_snv = k562_all_emvar_avg[(k562_all_emvar_avg['emVar'] == True) & 
                                    (k562_all_emvar_avg['cell_type'] == 'K562') & 
                                    (~k562_all_emvar_avg['id'].isin(k562_emvar_indel_avg['id'].tolist()))].drop_duplicates(subset='A_log2FC')
hepg2_emvar_snv = hepg2_all_emvar_avg[(hepg2_all_emvar_avg['emVar'] == True) & 
                                      (hepg2_all_emvar_avg['cell_type'] == 'HEPG2') & 
                                      (~hepg2_all_emvar_avg['id'].isin(hepg2_emvar_indel_avg['id'].tolist()))].drop_duplicates(subset='A_log2FC')
sknsh_emvar_snv = sknsh_all_emvar_avg[(sknsh_all_emvar_avg['emVar'] == True) & 
                                      (sknsh_all_emvar_avg['cell_type'] == 'SKNSH') &
                                      (~sknsh_all_emvar_avg['id'].isin(sknsh_emvar_indel_avg['id'].tolist()))].drop_duplicates(subset='A_log2FC')

In [ ]:
# calculate Kolmogorov-Smirnov test for each cell type
# k562
k_ks = stats.ks_2samp(
    [abs(i) for i in k562_emvar_indel_avg['Log2Skew']], # k562 indel skew
    [abs(i) for i in k562_emvar_snv['Log2Skew']] # k562 snv skew
)
print(f'the p-value of the kolmogorov-smirnov test i: {k_ks.pvalue:.2e}')
# hepg2
h_ks = stats.ks_2samp(
    [abs(i) for i in hepg2_emvar_indel_avg['Log2Skew']], # hepg2 indel skew
    [abs(i) for i in hepg2_emvar_snv['Log2Skew']]
)
print(f'the p-value of the kolmogorov-smirnov test i: {h_ks.pvalue:.2e}')
# sknsh
s_ks = stats.ks_2samp(
    [abs(i) for i in sknsh_emvar_indel_avg['Log2Skew']], # hepg2 indel skew
    [abs(i) for i in sknsh_emvar_snv['Log2Skew']]
)
print(f'the p-value of the kolmogorov-smirnov test i: {s_ks.pvalue:.2e}')

In [ ]:
# plot absolute distribution of skews for SNV emVars and indel emVars
# set up plot #
f = mpl.figure(dpi=300, figsize=(10,2))
gs = f.add_gridspec(1,3)
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
with sns.axes_style():
    ax_01 = f.add_subplot(gs[0,0])
    sns.histplot(
        x = [abs(i) for i in k562_emvar_snv['Log2Skew']],
        color = '#00A79D'
    )
    sns.histplot(
        [abs(i) for i in k562_emvar_indel_avg['Log2Skew']],
        color = '#bafffb'
    )
    mpl.xlabel('K562 Absolute Skew')
    mpl.annotate(f'p = {k_ks.pvalue:.2e}', xy=(4, 1000), fontsize=8)
with sns.axes_style():
    ax_02 = f.add_subplot(gs[0,1])
    sns.histplot(
        x = [abs(i) for i in hepg2_emvar_snv['Log2Skew']],
        color = '#FBB040'
    )
    sns.histplot(
        x = [abs(i) for i in hepg2_emvar_indel_avg['Log2Skew']],
        color = '#feefd9'
    )
    mpl.xlabel('HepG2 Absolute Skew')
    mpl.annotate(f'p = {h_ks.pvalue:.2e}', xy=(4, 1000), fontsize=8)
    mpl.ylabel('')
with sns.axes_style():
    ax_03 = f.add_subplot(gs[0,2])
    sns.histplot(
        x = [abs(i) for i in sknsh_emvar_snv['Log2Skew']],
        color = '#ED1C24'
    )
    sns.histplot(
        x = [abs(i) for i in sknsh_emvar_indel_avg['Log2Skew']],
        color = '#fbd2d3'
    )
    mpl.xlabel('SKNSH Absolute Skew')
    mpl.annotate(f'p = {s_ks.pvalue:.2e}', xy=(4, 1000), fontsize=8)
    mpl.ylabel('')
sns.despine()
mpl.savefig('../analysis/empirical_abs_emvar_skew_with_KS_pval.pdf')